# Chunking

## Overview

### Problem

Many documents (legal contracts, technical manuals, books) are significantly larger than any LLM's **Context Window** or the input size for _Embedding Models_.

Imagine that we want an LLM to answer questions based on a 6 500-page books. In total: 3,000 pages. A standard paperback page holds approximately 250 to 300 words. One word is generally equivalent to 1.33 tokens (often summarized as 75 words per 100 tokens). Calculating that for our knowledge base, we have: `1,200,000` tokens. More than most LLMs can handle.

Even much earlier than the theoretical limit, the LLM might experience **Context Rot**; performance degradation due to length of input.

[Amazon Science Research shows](https://www.amazon.science/publications/context-length-alone-hurts-llm-performance-despite-perfect-retrieval) that LLMs are best at recalling information located at the very beginning or the very end of a prompt; but not when it is buried in the middle of a massive block of text. This is known as the **"Lost in the Middle" Phenomena**.

**Cost** is calculated per input token, as well as per output token. The cost of the 1001th token recalculates the 1,000 previous tokens to produce it! 

This also means **Latency**; generation gets slower as tokens increase.

### Solution

This is why RAG systems work with **chunks** - smaller pieces of documents that can be independently retrieved based on relevance to a query.

A common **ingestion pipeline** works as follows: we split data into chunks, collect metadata fields we can attach to each chunk, and insert the resulting records into our Chroma collection. Chroma will automatically embed the chunks using the collection’s embedding function.

Without chunking:

```mermaid
flowchart LR
  D[/Document/]
  D1[Embedding Model]
  D --> D1 --> E1[/Vector /]
```

With chunking:

```mermaid
flowchart LR
  D[/Document/]
  D1[Embedding Model]
  D2[Embedding Model]
  D3[Embedding Model]
  D -- Split --> C1[/Chunk 1/] --> D1 --> E1[/Vector 1/]
  D -- Split --> C2[/Chunk 2/] --> D2 --> E2[/Vector 2/]
  D -- Split --> C3[/Chunk 3/] --> D3 --> E3[/Vector 3/]
```

## Setup

We reuse Chroma's in-memory client from [14_chromadb.ipynb](14_chromadb.ipynb).

On first run, Chroma downloads `all-MiniLM-L6-v2` (~79 MB) locally.

Docs: [chromadb](https://docs.trychroma.com/) · [langchain-text-splitters](https://docs.langchain.com/oss/python/integrations/splitters) · [pymupdf4llm](https://docs.pdf4llm.com/python/api/to_markdown) · [Chroma chunking guide](https://docs.trychroma.com/guides/build/chunking)

In [1]:
from pathlib import Path

import chromadb  # https://docs.trychroma.com/
from IPython.display import Markdown, display
from langchain_text_splitters import (  # https://docs.langchain.com/oss/python/integrations/splitters
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

from chunking_utils import (
    assign_expansion_metadata,
    expand_context,
    mrr,
    recall_at_k,
    to_chroma_records,
)

## Load

Real RAG pipelines often start with PDFs, not ready-made markdown. [pymupdf4llm](https://docs.pdf4llm.com/python/api/to_markdown) converts a PDF to markdown so the same chunking tools can process it. But first, we'll have to download it.

### Donwload Data

This sample pdf is titled *The Evolution of the Word Processor* — hosted on GitHub.

In [2]:
pdf_file_url = "https://github.com/docling-project/docling/blob/main/tests/data/pdf/multi_page.pdf"

# GitHub blob links need the raw URL for downloading
pdf_raw_url = pdf_file_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
print(pdf_raw_url)

https://raw.githubusercontent.com/docling-project/docling/main/tests/data/pdf/multi_page.pdf


In [3]:
import requests

SOURCE_NAME = "multi_page.pdf"
DOCUMENT_TITLE = "The Evolution of the Word Processor"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
source = data_dir / SOURCE_NAME

if not source.exists():
    response = requests.get(pdf_raw_url)
    response.raise_for_status()
    with open(source, "wb") as f:
        f.write(response.content)

### Parse PDF into Markdown with PDF4LLM

Extract clean, structured content from any document — ready for LLMs, RAG pipelines, and AI applications.

[PDF4LLM](https://docs.pdf4llm.com/) gives you something you can actually use — [clean Markdown](https://docs.pdf4llm.com/python/guides/extract-Markdown), [structured JSON](https://docs.pdf4llm.com/python/guides/extract-JSON), or [plain text](https://docs.pdf4llm.com/python/guides/extract-Text), with:

1. reading order preserved
2. tables intact
3. and images handled

..in a single function call.

In [4]:
import pymupdf4llm

base_extraction_options = {
    "footer": False,
    "header": False,
    "show_progress": True,
}

md = pymupdf4llm.to_markdown(
    source,
    **base_extraction_options,
    dpi=200,  # image resolution when write_images=True
    # pages=[0, 1, 2, 3, 4],   # first five pages only
    # page_chunks=True,          # return per-page dictionaries
    # write_images=True,         # extract images to disk
    # image_path="assets/",      # image output directory
    # image_format="png",        # image format
)

Parsing 5 pages of 'data/multi_page.pdf'...


100%|██████████| 5/5 [00:01<00:00,  3.03it/s]

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=2/3.
OCR on page.number=3/4.



In [5]:
print(f"Converted {len(md):,} characters · {md.count('##')} sections")

Converted 9,615 characters · 11 sections


Inspect the markdown (first 900 characters):

In [6]:
print(md[:900])

## **The Evolution of the Word Processor** 

The concept of the word processor predates modern computers and has evolved through several technological milestones. 

## **Pre-Digital Era (19th - Early 20th Century)** 

The origins of word processing can be traced back to the invention of the typewriter in the mid-19th century. Patented in 1868 by Christopher Latham Sholes, the typewriter revolutionized written communication by enabling people to produce legible, professional documents more efficiently than handwriting. 

During this period, the term "word processing" didn't exist, but the typewriter laid the groundwork for future developments. Over time, advancements such as carbon paper (for copies) and the electric typewriter (introduced by IBM in 1935) improved the speed and convenience of document creation. 

## **The Birth of Word Processing (1960s - 1970s)** 

The term "word process


You could also render it nicely:

In [7]:
# Uncomment to display the markdown
# display(Markdown(md[:900]))

Save the extracted markdown:

In [8]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
md_path = data_dir / f"{SOURCE_NAME}.md"
if not md_path.exists():
    with md_path.open("w", encoding="utf-8") as f:
        f.write(md)
else:
    print(f"Markdown file already exists: {md_path.resolve()}")

# print(f"Saved extracted markdown to: {md_path.resolve()}")

## Chunk and ingest

In [9]:
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="pdf-chunks",  # one named index inside this client
)

### Text Splitters

[**Text splitters**](https://docs.langchain.com/oss/python/integrations/splitters/index) break large docs into smaller chunks that will be retrievable individually and fit within model context window limit.

Chunking forces a trade-off:

- chunks need to be small enough to match specific queries
- but large enough to be self-contained and meaningful

The right boundaries depend on what we’re chunking. A novel has different natural units than an API reference. Code has different logical boundaries than an email thread.

In our case (PDF document) we will apply two splitting methods:

1. Document structured-based: [`MarkdownHeaderTextSplitter`](https://python.langchain.com/docs/modules/data_connection/document_transformers/markdown_header_metadata_splitter) — split at `#` / `##` / `###`
2. Text structured-based: [`RecursiveCharacterTextSplitter`](https://python.langchain.com/docs/modules/data_connection/document_transformers/text_splitter#recursivecharactertextsplitter) — splitting paragraphs, sentences, ..etc.

#### 1. Document structure-based splitting

Some documents have an inherent structure, such as HTML, Markdown, or JSON files. In these cases, it’s beneficial to split the document based on its structure, as it often naturally groups semantically related text. Key benefits of structure-based splitting:

- Preserves the logical organization of the document
- Maintains context within each chunk
- Can be more effective for downstream tasks like retrieval or summarization

**Available text splitters**:

- [Split Markdown](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter): Split based on headers (e.g., `#`, `##`, `###`)
- [Split JSON](https://docs.langchain.com/oss/python/integrations/splitters/recursive_json_splitter): Split by object or array elements
- [Split code](https://docs.langchain.com/oss/python/integrations/splitters/code_splitter):  Split by functions, classes, or logical blocks
- [Split HTML](https://docs.langchain.com/oss/python/integrations/splitters/split_html): Split using tags



When chunking Markdown files, we can take advantage of their structure. For example, we can split by headers - try to split by `h2` headers, and recursively try inner headers.

LangChain’s `MarkdownHeaderTextSplitter` splits by section and captures the header hierarchy as metadata.

In [10]:
# Step 1: split at markdown headers (metadata keys = h1, h2, h3)
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ],
)

#### 2. Text structure-based splitting

Text is naturally organized into hierarchical units such as paragraphs, sentences, and words. We can leverage this inherent structure to inform our splitting strategy, creating split that maintain natural language flow, maintain semantic coherence within split, and adapts to varying levels of text granularity. LangChain’s `RecursiveCharacterTextSplitter` implements this concept:

- The [`RecursiveCharacterTextSplitter`](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) attempts to keep larger units (e.g., paragraphs) intact.
- If a unit exceeds the chunk size, it moves to the next level (e.g., sentences).
- This process continues down to the word level if necessary.

The `chunk_overlap` creates redundancy that helps preserve context across boundaries. The downside: you’re storing and embedding duplicate content.

In [11]:
# Step 2: constrain chunk size within each header group
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # target tokens/chars per chunk
    chunk_overlap=30,    # overlap keeps context across boundaries
)

#### 3. Semantic splitting

This method is more computationally expensive but can produce more coherent chunks when documents lack clear structural markers:

- Embed sentences or paragraphs, compute similarity between adjacent segments, and place chunk boundaries where similarity drops (indicating a topic shift).
- Alternatively use an LLM to decide where the boundaries are.

We won't use it in this tutorial.

### Save metadata for later Context Expansion

To provide more complete and coherent information to downstream models or applications:

When storing each chunk, we track its position within the section (`section_id` and `chunk_index`) as `metadata`. That's what the `assign_expansion_metadata()` function do.

Later, when retrieving a chunk, we can quickly fetch the **previous and next** chunks within the same section.

In [12]:
file_meta = {
    "source": SOURCE_NAME,
    "document_title": DOCUMENT_TITLE,
}

md_splits = md_splitter.split_text(md)
for split in md_splits:
    split.metadata = {**file_meta, **split.metadata}

# split_documents (not split_text) keeps header metadata on sub-chunks
all_chunks = char_splitter.split_documents(md_splits)
all_chunks = assign_expansion_metadata(all_chunks)

Let's look at the first 3 chunks:

In [13]:
for ch in all_chunks[:3]:
    print(ch.page_content)
    for k, v in ch.metadata.items():
        print(f"\t{k}: {v}")
    print()

The concept of the word processor predates modern computers and has evolved through several technological milestones.
	source: multi_page.pdf
	document_title: The Evolution of the Word Processor
	h2: **The Evolution of the Word Processor**
	section_id: multi_page.pdf::**The Evolution of the Word Processor**
	section_index: 0
	chunk_index: 0
	chunk_count: 1

The origins of word processing can be traced back to the invention of the typewriter in the mid-19th century. Patented in 1868 by Christopher Latham Sholes, the typewriter revolutionized written
	source: multi_page.pdf
	document_title: The Evolution of the Word Processor
	h2: **Pre-Digital Era (19th - Early 20th Century)**
	section_id: multi_page.pdf::**Pre-Digital Era (19th - Early 20th Century)**
	section_index: 1
	chunk_index: 0
	chunk_count: 4

revolutionized written communication by enabling people to produce legible, professional documents more efficiently than handwriting.
	source: multi_page.pdf
	document_title: The Evolutio

### Insert as records into ChromaDB

Since splitters come from LangChain, they produced a `Document` instance, which we'll need to serialize to a format that the function `.add()` accepts. This is what the `to_chroma_records()` is here for:

In [14]:
chunk_ids, chunk_texts, chunk_metas = to_chroma_records(all_chunks)

collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    metadatas=chunk_metas,
)

print(f"Ingested {collection.count()} chunks.")

Ingested 69 chunks.


## Query the collection

Semantic search embeds the query and returns the nearest chunks ([Chroma querying](https://docs.trychroma.com/docs/querying-collections/query-and-get)).

In [15]:
query = "When was Microsoft Word launched?"

results = collection.query(
    query_texts=[query],           # text to embed and search with
    n_results=3,                   # top-k nearest neighbors
    include=["documents", "metadatas"],
)

for doc_id, doc, meta in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
):
    section = meta.get("h2") or meta.get("h1", "")
    print(f"{doc_id} | {section}")
    display(Markdown(doc))
    print()

multi_page.pdf::s03::c03 | **The Rise of Personal Computers (1980s)**


[**The Rise of Personal Computers (1980s)**]

- **Microsoft Word (1983)** : Microsoft launched Word for MS-DOS in 1983, introducing a graphical user interface (GUI) and mouse support. Over the years, Microsoft Word became the industry standard


multi_page.pdf::s04::c01 | **The Modern Era (1990s - Present)**


[**The Modern Era (1990s - Present)**]

- **Microsoft Office Suite** : Microsoft continued to dominate with its Office Suite, integrating Word with other productivity tools like Excel and PowerPoint.


multi_page.pdf::s03::c04 | **The Rise of Personal Computers (1980s)**


[**The Rise of Personal Computers (1980s)**]

became the industry standard for word processing.

### Documents in Vector Store are Chunks

Now that we have ingested chunks, we can imagine them as points in the ChromaDB vector store.

But what is considered a `Document` is really a Chunk, a small piece, which may or **may not be enough to answer a query in full**.

![](../assets/vector_db_query.png){height=300}

### Context expansion

A retrieved chunk may be incomplete on its own — a definition in one paragraph and the formula in the next.

The `expand_context()` functoin fetches neighboring chunks from the same section using `section_id` and `chunk_index` stored at ingestion time.

In [16]:
expansion_query = "What was the IBM MT/ST?"

hit = collection.query(
    query_texts=[expansion_query],
    n_results=1,
    include=["documents", "metadatas"],
)

hit_id = hit["ids"][0][0]
hit_meta = hit["metadatas"][0][0]

print("Retrieved chunk only:\n")
display(Markdown(hit["documents"][0][0]))

print("\nExpanded context (±1 neighbor):\n")
for doc_id, doc, _meta in expand_context(collection, hit_meta, window=1):
    label = " ← hit" if doc_id == hit_id else ""
    print(f"--- {doc_id}{label} ---")
    display(Markdown(doc))
    print()

Retrieved chunk only:



[**The Birth of Word Processing (1960s - 1970s)**]

- **IBM MT/ST (Magnetic Tape/Selectric Typewriter)** : Introduced in 1964, this machine combined IBM's Selectric typewriter with magnetic tape storage. It allowed users to record, edit, and replay


Expanded context (±1 neighbor):

--- multi_page.pdf::s02::c01 ---


[**The Birth of Word Processing (1960s - 1970s)**]

were not software programs but rather standalone machines.


--- multi_page.pdf::s02::c02 ← hit ---


[**The Birth of Word Processing (1960s - 1970s)**]

- **IBM MT/ST (Magnetic Tape/Selectric Typewriter)** : Introduced in 1964, this machine combined IBM's Selectric typewriter with magnetic tape storage. It allowed users to record, edit, and replay


--- multi_page.pdf::s02::c03 ---


[**The Birth of Word Processing (1960s - 1970s)**]

to record, edit, and replay typed content—an early example of digital text storage.

## Evaluation

Create a set of test queries with ground truth: each query maps to the chunk(s) that should be retrieved for it. Then, we'll use two metrics:

- **Recall@k**: Of your test queries, what percentage have the correct chunk in the top `k` results?
- **Mean Reciprocal Rank (MRR)** - Where does the first correct chunk appear? (Higher is better)

In [17]:
test_queries = [
    {
        "query": "When was Microsoft Word launched?",
        "expected_ids": ["multi_page.pdf::s03::c03"],
    },
    {
        "query": "What is Google Docs and when was it introduced?",
        "expected_ids": ["multi_page.pdf::s04::c03"],
    },
    {
        "query": "What is LaTeX used for in academic writing?",
        "expected_ids": ["multi_page.pdf::s06::c01"],
    },
    {
        "query": "What is real-time collaboration in word processors?",
        "expected_ids": ["multi_page.pdf::s07::c06"],
    },
    {
        "query": "What was WordStar?",
        "expected_ids": ["multi_page.pdf::s03::c01"],
    },
]

K = 5
eval_results = collection.query(
    query_texts=[case["query"] for case in test_queries],
    n_results=K,
)

for i, case in enumerate(test_queries):
    ids = eval_results["ids"][i]
    print(
        f"{case['query'][:50]:50}  "
        f"recall={recall_at_k(ids, case['expected_ids'], K):.2f}  "
        f"mrr={mrr(ids, case['expected_ids']):.2f}"
    )

When was Microsoft Word launched?                   recall=1.00  mrr=1.00
What is Google Docs and when was it introduced?     recall=1.00  mrr=1.00
What is LaTeX used for in academic writing?         recall=1.00  mrr=1.00
What is real-time collaboration in word processors  recall=1.00  mrr=1.00
What was WordStar?                                  recall=1.00  mrr=1.00


### Interpreting results

If you see:

* Low recall (the correct chunks are not in the top-k results) - try smaller chunks, with more overlap between them.
* Correct chunks rank low - add context to the chunks themselves and leverage metadata filtering
* Duplicate results - decrease chunk overlap
* Irrelevant matches - try larger chunks, structure-aware chunking, or semantic-aware chunking.

## Takeaways

- **Chunking** lets RAG systems search a long PDF one passage at a time
- LangChain's **`split_documents`** pipeline preserves header metadata through size-based splitting
- **Metadata** (`section_id`, `chunk_index`) supports context expansion at retrieval time
- **Evaluation** with recall@k and MRR helps compare chunking strategies before wiring up an LLM